# Retinal Baseline Validation

## Purpose
This notebook validates a trained retinal disease classifier checkpoint after training has completed. It is separate from the training notebook so you can inspect model quality, recompute evaluation metrics, and manually review predictions without mixing validation logic into the training workflow.

## Requirements
- Google Colab runtime with GPU enabled
- Google Drive mounted in the notebook
- Repo available at `MyDrive/retina_project/repo`
- Dataset available at `MyDrive/retina_project/data`
- A completed training run under `MyDrive/retina_project/runs/`
- Required run artifacts: `best_model.pt`, `label_map.json`, `run_config.json`, `metrics.json`

## Expected Drive Layout
```text
MyDrive/
  retina_project/
    repo/
      training/
      colab/
      ...
    data/
      splits/
        test.csv
      raw/
        original/
          ...
    runs/
      baseline_efficientnet_b0_provisional/
        best_model.pt
        metrics.json
        label_map.json
        run_config.json
        ...
```

## What This Notebook Does
- Mounts Google Drive
- Installs validation dependencies
- Verifies the repo, run artifacts, manifest, and raw image paths
- Loads the saved checkpoint and rebuilds the correct model
- Recomputes test-set metrics from the saved model
- Renders a confusion matrix in notebook output
- Displays one representative test image per class with prediction details

## Outputs
This notebook is primarily for inspection and does not retrain the model. It will display:
- recomputed accuracy, macro F1, weighted F1
- per-class precision / recall / F1
- a fresh confusion matrix
- one sample prediction per class with top-3 probabilities

## Important Caveats
- If the run was trained in provisional mode, the validation reflects unreviewed data
- This notebook validates a research baseline, not a clinical diagnostic system
- Results are only meaningful if the run directory, test manifest, and raw image files match the original training setup


In [4]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
!pip install -q torch torchvision timm scikit-learn pandas pillow matplotlib

In [6]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU not detected. In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> GPU, reconnect, and rerun the notebook.'
    )

print('GPU is available.')
print('Device:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [7]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/retina_project')
REPO_ROOT = PROJECT_ROOT / 'repo'
DATA_ROOT = PROJECT_ROOT
RUN_DIR = PROJECT_ROOT / 'runs' / 'baseline_efficientnet_b0_provisional'
TEST_CSV = DATA_ROOT / 'data' / 'splits' / 'test.csv'
RAW_ROOT = DATA_ROOT / 'data' / 'raw' / 'original'

print(PROJECT_ROOT)
print(REPO_ROOT)
print(DATA_ROOT)
print(RUN_DIR)
print(TEST_CSV)
print(RAW_ROOT)

In [8]:
import json
import pandas as pd

required_paths = {
    'repo_root': REPO_ROOT,
    'training_config': REPO_ROOT / 'training' / 'config.py',
    'run_dir': RUN_DIR,
    'checkpoint': RUN_DIR / 'best_model.pt',
    'metrics': RUN_DIR / 'metrics.json',
    'label_map': RUN_DIR / 'label_map.json',
    'run_config': RUN_DIR / 'run_config.json',
    'test_csv': TEST_CSV,
    'raw_root': RAW_ROOT,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
for name, path in required_paths.items():
    print(f'{name}: {path} -> {path.exists()}')

if missing:
    raise FileNotFoundError('Missing required paths: ' + ', '.join(missing))

test_preview = pd.read_csv(TEST_CSV, nrows=5)
print('\nTest CSV columns:', list(test_preview.columns))
print(test_preview[['image_id', 'file_path', 'class_name']])

sample_paths = [DATA_ROOT / path for path in test_preview['file_path'].tolist()]
for path in sample_paths:
    print(f'sample_image: {path} -> {path.exists()}')

if not all(path.exists() for path in sample_paths):
    raise FileNotFoundError('At least one sample image path from the test manifest does not resolve under DATA_ROOT.')

with open(RUN_DIR / 'run_config.json', 'r', encoding='utf-8') as handle:
    run_config = json.load(handle)
with open(RUN_DIR / 'label_map.json', 'r', encoding='utf-8') as handle:
    label_map = json.load(handle)

print('\nSaved run config:')
print(json.dumps(run_config, indent=2))
print('\nLabel map:')
print(label_map)


In [9]:
import json
import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

class_to_index = {key: int(value) for key, value in label_map.items()}
index_to_class = {value: key for key, value in class_to_index.items()}
class_names = [index_to_class[index] for index in sorted(index_to_class)]

image_size = int(run_config['img_size'])
model_name = run_config['model']

eval_transform = transforms.Compose([
    transforms.Resize(int(image_size * 1.15)),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location=device)
model = timm.create_model(model_name, pretrained=False, num_classes=len(class_names))
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

test_df = pd.read_csv(TEST_CSV)
test_df = test_df[test_df['class_name'].isin(class_to_index.keys())].copy()
test_df['resolved_path'] = test_df['file_path'].map(lambda value: str((DATA_ROOT / value).resolve()))
test_df['label'] = test_df['class_name'].map(class_to_index)

print('test rows used:', len(test_df))
print(test_df['class_name'].value_counts().to_dict())


In [10]:
from torch.utils.data import DataLoader, Dataset

class ValidationDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row['resolved_path']).convert('RGB')
        image = self.transform(image)
        return image, int(row['label'])

dataset = ValidationDataset(test_df, eval_transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        predictions = torch.argmax(logits, dim=1).cpu().tolist()
        all_predictions.extend(predictions)
        all_labels.extend(labels.tolist())

report = classification_report(all_labels, all_predictions, target_names=class_names, output_dict=True, zero_division=0)
matrix = confusion_matrix(all_labels, all_predictions, labels=list(range(len(class_names))))

recomputed_metrics = {
    'accuracy': float(accuracy_score(all_labels, all_predictions)),
    'macro_f1': float(f1_score(all_labels, all_predictions, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(all_labels, all_predictions, average='weighted', zero_division=0)),
    'per_class': {
        class_name: {
            'precision': float(report[class_name]['precision']),
            'recall': float(report[class_name]['recall']),
            'f1': float(report[class_name]['f1-score']),
            'support': int(report[class_name]['support']),
        }
        for class_name in class_names
    },
    'confusion_matrix': matrix.tolist(),
}

print(json.dumps(recomputed_metrics, indent=2))


In [11]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(matrix, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(image, ax=ax)
ax.set(
    xticks=np.arange(len(class_names)),
    yticks=np.arange(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    ylabel='True label',
    xlabel='Predicted label',
    title='Recomputed Test Confusion Matrix',
)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')

threshold = matrix.max() / 2.0 if matrix.size else 0.0
for row_index in range(matrix.shape[0]):
    for column_index in range(matrix.shape[1]):
        ax.text(
            column_index,
            row_index,
            str(matrix[row_index, column_index]),
            ha='center',
            va='center',
            color='white' if matrix[row_index, column_index] > threshold else 'black',
        )

fig.tight_layout()
plt.show()


In [12]:
with open(RUN_DIR / 'metrics.json', 'r', encoding='utf-8') as handle:
    saved_metrics = json.load(handle)

print('Saved metrics snapshot:')
print(json.dumps(saved_metrics, indent=2))


In [13]:
sample_rows = (
    test_df.sort_values(['class_name', 'image_id'])
    .groupby('class_name', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print(sample_rows[['image_id', 'class_name', 'resolved_path']])


In [14]:
fig, axes = plt.subplots(len(sample_rows), 1, figsize=(8, 4 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    image = Image.open(row['resolved_path']).convert('RGB')
    x = eval_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu()

    top_values, top_indices = torch.topk(probs, k=min(3, len(class_names)))
    predicted_index = int(top_indices[0])
    predicted_label = index_to_class[predicted_index]
    confidence = float(top_values[0])
    top3 = ', '.join(
        f"{index_to_class[int(idx)]}: {float(val):.3f}"
        for val, idx in zip(top_values, top_indices)
    )

    ax.imshow(image)
    ax.axis('off')
    ax.set_title(
        f"true={row['class_name']} | pred={predicted_label} | conf={confidence:.3f}\n{top3}",
        fontsize=11,
    )

plt.tight_layout()
plt.show()
